# LGBM Model Subclass

The LGBM model trains 1 model per quantile, using 28 days of sales as flat features. It predicts autoregressively, feeding the median prediction back as a new input at each step. This notebook reuses the LGBM logic in final_presentation_notebook.ipynb and makes the following changes:
1. add richer tabular features so that the model can use context properly (TODO)

In [ ]:
# !pip install joblib

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)


In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from base_model import BaseModel

In [ ]:
# Step 1: class skeleton - every BaseModel subclass must implement these methods
class LightGBM(BaseModel):

    @property
    def model_name(self):
        return "lightgbm"
    
    def preprocess(self): pass
    def train(self): pass
    def predict(self): pass


In [31]:
# Step 2: model-specific preprocessing
def preprocess(self):
    def _build_features(df):
        df = df.sort_values(["id", "d_num"]).copy()
        g = df.groupby("id")["sales"]
        # past sales at fixed offsets. Removed lag 28 to avoid losing too many rows
        for lag in [1, 7, 14]:   
            df[f"lag_{lag}"] = g.shift(lag)
        # smooth out noise and capture the short-term trend
        for w in [7, 28]:
            df[f"roll_mean_{w}"] = g.shift(1).rolling(w).mean() 
        # encode Mon-Sun as a cycle so that day 7 is close to day 1 
        df["dow_sin"] = np.sin(2 * np.pi * df["wday"] / 7).astype(np.float32)   
        df["dow_cos"] = np.cos(2 * np.pi * df["wday"] / 7).astype(np.float32)
        # raw day number lets the model learn long-term trends
        df["trend"]   = df["d_num"].astype(np.float32)
        # drop any row with missing target (incomplete windows)
        df["target"]  = g.shift(-1)   
        feat_cols = ["lag_1","lag_7","lag_14","roll_mean_7","roll_mean_28"]
        # early rows where lags/rolling means are unavailable are filled wth 0 
        df[feat_cols] = df[feat_cols].fillna(0)
        return df.dropna(subset=["target"]).reset_index(drop=True)

    self.train_processed = _build_features(self.train_raw)
    self.val_processed   = _build_features(self.val_raw)
    self.test_processed  = _build_features(self.test_raw)

LightGBM.preprocess = preprocess

In [32]:
m = LightGBM()
m.load_and_split_data()
m.preprocess()

print(m.train_processed.shape)  # expected 54028280 rows (30490 items * (1773 training days - 1 day target))
print(m.train_processed[["id","d_num","lag_1","lag_14","roll_mean_7","target"]].sample(5))

Loaded cached data splits.
(54028280, 33)
                                     id  d_num  lag_1  lag_14  roll_mean_7  \
27166752  HOBBIES_1_100_CA_2_evaluation    221    0.0     1.0     0.571429   
24562664    FOODS_3_777_CA_2_evaluation    973    0.0     0.0     0.000000   
15595597    FOODS_3_268_CA_2_evaluation    226    0.0     0.0     0.000000   
18893234    FOODS_3_454_CA_3_evaluation    171    0.0     0.0     0.000000   
16746971    FOODS_3_333_CA_1_evaluation   1572    1.0     2.0     2.857143   

          target  
27166752     0.0  
24562664     0.0  
15595597     0.0  
18893234     0.0  
16746971     0.0  


In [ ]:
# raw_rows     = len(m.train_raw)
# tgt_dropped  = m.train_raw.groupby("id")["sales"].shift(-1).isna().sum()

# print(f"Raw rows:         {raw_rows:,}")
# print(f"Dropped (target): {tgt_dropped:,}")
# print(f"Expected:         {raw_rows - tgt_dropped:,}")
# print(f"Actual:           {len(m.train_processed):,}")

Raw rows:         54,058,770
Dropped (target): 30,490
Expected:         54,028,280
Actual:           54,028,280


Train 1 LGBM model per quantile (9 in total) with objective = quantile. 
- Each model is input the same feature matrix X but is optimised to predict a different part of the distribution (specified by setting alpha)
- Models are saved as .txt files to be reloaded in predict()

In [ ]:
# Step 3: model training

def train(self):
    FEAT_COLS = [
        "lag_1","lag_7","lag_14",
        "roll_mean_7","roll_mean_28",
        "dow_sin","dow_cos","trend",
        "wday","month","sell_price","is_available"
    ]
    X = self.train_processed[FEAT_COLS].values.astype(np.float32)
    y = self.train_processed["target"].values.astype(np.float32)

    self.models = {}
    for q in self.QUANTILES:
        dtrain = lgb.Dataset(X, y)
        params = {
            "objective": "quantile", "alpha": q,
            "learning_rate": 0.05, "num_leaves": 64, "verbose": -1
        }
        self.models[q] = lgb.train(params, dtrain, num_boost_round=200)
        self.models[q].save_model(
            os.path.join(self.output_dir, f"{self.model_name}_q{q}.txt")
        )
    print(f"Saved {len(self.models)} models to {self.output_dir}/")

LightGBM.train = train

In [34]:
m.train()

KeyboardInterrupt: 

In [ ]:
# Step 4: model inference
def predict(self):
    FEAT_COLS = [
        "lag_1","lag_7","lag_14",
        "roll_mean_7","roll_mean_28",
        "dow_sin","dow_cos","trend",
        "wday","month","sell_price","is_available"
    ]
    # load 9 saved models from disk
    models = {
        q: lgb.Booster(model_file=os.path.join(self.output_dir, f"{self.model_name}_q{q}.txt"))
        for q in self.QUANTILES
    }

    ctx = self.test_processed[
        (self.test_processed["d_num"] >= 1886) &
        (self.test_processed["d_num"] <= 1913)
    ].copy()

    rows = []
    for item_id, item_ctx in ctx.groupby("id"):
        item_ctx = item_ctx.sort_values("d_num")
        buf      = item_ctx["sales"].values[-28:].tolist()
        last     = item_ctx.iloc[-1]

        # Autoregressively forecast 28 steps per item
        for h in range(1, 29):
            dow = int((last["wday"] + h - 1) % 7)
            # build a feature vector from a rolling sales buffer
            x = np.array([[
                buf[-1], buf[-7], buf[-14],
                np.mean(buf[-7:]), np.mean(buf),
                np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7),
                last["d_num"] + h,
                dow, last["month"], last["sell_price"], last["is_available"]
            ]], dtype=np.float32)

            rec = {"id": item_id, "day_ahead": h}
            preds_step = {}
            # predict all 9 quantiles
            for q in self.QUANTILES:
                p = max(0.0, float(models[q].predict(x)[0]))
                preds_step[q] = p
                rec[f"q{q}"] = p

            # push the median back into the buffer
            buf.append(preds_step[0.5])
            rows.append(rec)

    # Post-processing
    preds_df = pd.DataFrame(rows).sort_values(["id","day_ahead"]).reset_index(drop=True)
    q_cols = [f"q{q}" for q in self.QUANTILES]
    # clip negative predictions to 0
    preds_df[q_cols] = preds_df[q_cols].clip(lower=0)
    # enforce non-decreasing quantiles
    preds_df[q_cols] = np.maximum.accumulate(preds_df[q_cols].values, axis=1)

    out = os.path.join(self.output_dir, f"{self.model_name}_predictions.csv")
    preds_df.to_csv(out, index=False)
    print(f"Saved → {out}")
    return preds_df

LightGBM.predict = predict

In [ ]:
preds_df = m.predict()

print(preds_df.shape)           # expect (853720, 11) — 30490 items × 28 days
print(preds_df.columns.tolist())
print(preds_df.head(3))

q_cols = [f"q{q}" for q in m.QUANTILES]
assert (preds_df[q_cols].diff(axis=1).iloc[:, 1:] >= 0).all().all(), "quantiles not non-decreasing"
assert (preds_df[q_cols] >= 0).all().all(), "negative predictions"
print("Quantiles valid")